In [1]:
# ═══════════════════════════════════════════════════════════
# CELL 1: DATA LOADING & FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.impute import SimpleImputer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────
# LOAD RAW DATA
# ─────────────────────────────────────────────────
DS_PATH = "/kaggle/input/datasets/shifasoni/emosurv"  

fixed_df = pd.read_csv(f"{DS_PATH}/Fixed Text Typing Dataset.csv", sep=';')
free_df  = pd.read_csv(f"{DS_PATH}/Free Text Typing Dataset.csv", sep=';')
freq_df  = pd.read_csv(f"{DS_PATH}/Frequency Dataset.csv", sep=';')
info_df  = pd.read_csv(f"{DS_PATH}/Participants Information.csv", sep=';')

print("Raw data shapes:")
print(f"  Fixed text: {fixed_df.shape} — columns: {list(fixed_df.columns)}")
print(f"  Free text:  {free_df.shape}  — columns: {list(free_df.columns)}")
print(f"  Frequency:  {freq_df.shape}  — columns: {list(freq_df.columns)}")
print(f"  Participants: {info_df.shape}")

# ─────────────────────────────────────────────────
# COMBINE FIXED + FREE TEXT DATASETS
# ─────────────────────────────────────────────────
print(f"\nFixed text columns: {list(fixed_df.columns)}")
print(f"Free text columns: {list(free_df.columns)}")

uid_col = 'userId'
emo_col = 'emotionIndex'

timing_cols = ['D1U1', 'D1U2', 'D1D2', 'U1D2', 'U1U2', 'D1U3', 'D1D3']

# Normalize column names for free text
free_df = free_df.rename(columns={'userid': 'userId'})

# Tag source
fixed_df['textType'] = 'fixed'
free_df['textType'] = 'free'

# Combine
common_cols = [uid_col, emo_col, 'keyCode', 'keyDown', 'keyUp'] + timing_cols + ['textType']
typing_df = pd.concat([fixed_df[common_cols], free_df[common_cols]], ignore_index=True)

# Convert timing columns to numeric
for col in timing_cols + ['keyDown', 'keyUp']:
    typing_df[col] = pd.to_numeric(typing_df[col], errors='coerce')

print(f"\nUsing columns: userId='{uid_col}', emotionIndex='{emo_col}'")
print(f"Emotion values: {typing_df[emo_col].unique()}")
print(f"Total keystroke events: {len(typing_df)}")
print(f"Unique users: {typing_df[uid_col].nunique()}")
print(f"Timing columns found: {timing_cols}")

# ─────────────────────────────────────────────────
# COMPUTE PER-SESSION FEATURES
# ─────────────────────────────────────────────────
# Each session = one (user, emotion, textType) group.
# For each of 7 timing columns, compute 6 statistics.

def compute_session_features(group):
    features = {}
    for col in timing_cols:
        data = pd.to_numeric(group[col], errors='coerce').dropna()
        if len(data) < 3:
            for stat in ['mean', 'std', 'median', 'q25', 'q75', 'iqr']:
                features[f'{col}_{stat}'] = np.nan
        else:
            features[f'{col}_mean'] = data.mean()
            features[f'{col}_std'] = data.std()
            features[f'{col}_median'] = data.median()
            features[f'{col}_q25'] = data.quantile(0.25)
            features[f'{col}_q75'] = data.quantile(0.75)
            features[f'{col}_iqr'] = data.quantile(0.75) - data.quantile(0.25)
    
    # Derived features
    d1d2 = pd.to_numeric(group['D1D2'], errors='coerce').dropna()
    d1u1 = pd.to_numeric(group['D1U1'], errors='coerce').dropna()
    u1d2 = pd.to_numeric(group['U1D2'], errors='coerce').dropna()
    
    features['typing_speed'] = 1.0 / d1d2.mean() if len(d1d2) > 0 and d1d2.mean() > 0 else np.nan
    features['rhythm_cv'] = d1d2.std() / d1d2.mean() if len(d1d2) > 2 and d1d2.mean() > 0 else np.nan
    
    if len(d1u1) > 0 and len(u1d2) > 0 and u1d2.mean() != 0:
        features['dwell_flight_ratio'] = d1u1.mean() / (abs(u1d2.mean()) + 1e-8)
    else:
        features['dwell_flight_ratio'] = np.nan
    
    features['n_keystrokes'] = len(group)
    
    return pd.Series(features)

print("\nComputing per-session features...")
session_features = typing_df.groupby([uid_col, emo_col, 'textType']).apply(
    compute_session_features
).reset_index()

print(f"Sessions created: {len(session_features)}")
print(f"Features per session: {session_features.shape[1] - 3}")

# ─────────────────────────────────────────────────
# ADD FREQUENCY FEATURES (error rates)
# ─────────────────────────────────────────────────
freq_uid_col = [c for c in freq_df.columns if 'user' in c.lower()][0]
freq_emo_col = [c for c in freq_df.columns if 'emotion' in c.lower()][0]
freq_text_col = [c for c in freq_df.columns if 'text' in c.lower()][0]

freq_df['textType'] = freq_df[freq_text_col].map({'FI': 'fixed', 'FR': 'free'})

del_col = [c for c in freq_df.columns if 'del' in c.lower()][0]
left_col = [c for c in freq_df.columns if 'left' in c.lower()][0]
time_col = [c for c in freq_df.columns if 'time' in c.lower()][0]

freq_merge = freq_df[[freq_uid_col, freq_emo_col, 'textType', del_col, left_col, time_col]].copy()
freq_merge.columns = [uid_col, emo_col, 'textType', 'DelFreq', 'LeftFreq', 'TotTime']

for c in ['DelFreq', 'LeftFreq', 'TotTime']:
    freq_merge[c] = pd.to_numeric(freq_merge[c], errors='coerce')

freq_merge['error_ratio'] = freq_merge['DelFreq'].fillna(0) + freq_merge['LeftFreq'].fillna(0)

session_features = session_features.merge(freq_merge, on=[uid_col, emo_col, 'textType'], how='left')

print(f"\nAfter merging frequency data: {session_features.shape}")

# ─────────────────────────────────────────────────
# PREPARE FINAL DATASET
# ─────────────────────────────────────────────────
emotion_map = {'H': 'happy', 'S': 'sad', 'A': 'angry', 'C': 'calm', 'N': 'neutral'}
session_features['emotion'] = session_features[emo_col].map(emotion_map)

feature_cols = [c for c in session_features.columns 
                if c not in [uid_col, emo_col, 'textType', 'emotion']]

session_features = session_features.dropna(subset=['emotion'])

X = session_features[feature_cols].values.astype(np.float64)
y = session_features['emotion'].values
groups = session_features[uid_col].values

imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\n{'='*50}")
print(f"Final dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"{'='*50}")
print(f"\nEmotion distribution:")
for emo, count in sorted(Counter(y).items()):
    print(f"  {emo}: {count}")
print(f"\nUnique users: {len(np.unique(groups))}")
print(f"Chance level (5-class): {1/5:.1%}")
print(f"\nFeature columns ({len(feature_cols)}):")
print(f"  {feature_cols}")


Raw data shapes:
  Fixed text: (46871, 14) — columns: ['userId', 'emotionIndex', 'index', 'keyCode', 'keyDown', 'keyUp', 'D1U1', 'D1U2', 'D1D2', 'U1D2', 'U1U2', 'D1U3', 'D1D3', 'answer']
  Free text:  (28412, 15)  — columns: ['_id', 'userid', 'emotionIndex', 'index', 'keyCode', 'keyDown', 'keyUp', 'D1U1', 'D1U2', 'D1D2', 'U1D2', 'U1U2', 'D1U3', 'D1D3', 'answer']
  Frequency:  (478, 6)  — columns: ['User ID', 'textIndex', 'emotionIndex', 'delFreq', 'leftFreq', 'TotTime']
  Participants: (123, 9)

Fixed text columns: ['userId', 'emotionIndex', 'index', 'keyCode', 'keyDown', 'keyUp', 'D1U1', 'D1U2', 'D1D2', 'U1D2', 'U1U2', 'D1U3', 'D1D3', 'answer']
Free text columns: ['_id', 'userid', 'emotionIndex', 'index', 'keyCode', 'keyDown', 'keyUp', 'D1U1', 'D1U2', 'D1D2', 'U1D2', 'U1U2', 'D1U3', 'D1D3', 'answer']

Using columns: userId='userId', emotionIndex='emotionIndex'
Emotion values: ['N' 'H' 'C' 'A' 'S']
Total keystroke events: 75283
Unique users: 83
Timing columns found: ['D1U1', 'D1U2', 'D

In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 2: MODEL COMPARISON + FEATURE IMPORTANCE
# ═══════════════════════════════════════════════════════════

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Classes: {list(le.classes_)}")

def evaluate(X, y, groups, model_class, name, **kwargs):
    gkf = GroupKFold(n_splits=5)
    accs, f1s = [], []
    for train_idx, test_idx in gkf.split(X, y, groups):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X[train_idx])
        X_te = scaler.transform(X[test_idx])
        model = model_class(**kwargs)
        model.fit(X_tr, y[train_idx])
        pred = model.predict(X_te)
        accs.append(accuracy_score(y[test_idx], pred))
        f1s.append(f1_score(y[test_idx], pred, average='weighted'))
    print(f"  {name}: Acc={np.mean(accs):.3f}±{np.std(accs):.3f}  "
          f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")
    return np.mean(accs)

print("=" * 60)
print("RESULTS — Keystroke Emotion Detection (5-class)")
print("=" * 60)

acc_lr = evaluate(X, y_encoded, groups, LogisticRegression,
                  "LogReg (baseline)", max_iter=1000, class_weight='balanced', random_state=42)
acc_svm = evaluate(X, y_encoded, groups, SVC,
                   "SVM (RBF)", kernel='rbf', class_weight='balanced', C=1.0, random_state=42)
acc_rf = evaluate(X, y_encoded, groups, RandomForestClassifier,
                  "Random Forest", n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
acc_gb = evaluate(X, y_encoded, groups, GradientBoostingClassifier,
                  "Gradient Boost", n_estimators=200, max_depth=5, random_state=42)

print(f"\nChance level: {1/len(le.classes_):.3f} ({1/len(le.classes_):.1%})")
print(f"Best accuracy: {max(acc_lr, acc_svm, acc_rf, acc_gb):.3f}")

# ── 3-class ──
ks_sentiment_map = {
    'happy': 'positive', 'calm': 'positive',
    'angry': 'negative', 'sad': 'negative',
    'neutral': 'neutral'
}
y_3 = np.array([ks_sentiment_map[e] for e in y])
le_3 = LabelEncoder()
y_3_enc = le_3.fit_transform(y_3)

print(f"\n{'='*60}")
print("RESULTS — Keystroke Emotion Detection (3-class)")
print("=" * 60)

acc3_lr = evaluate(X, y_3_enc, groups, LogisticRegression,
                   "LogReg (baseline)", max_iter=1000, class_weight='balanced', random_state=42)
acc3_svm = evaluate(X, y_3_enc, groups, SVC,
                    "SVM (RBF)", kernel='rbf', class_weight='balanced', C=1.0, random_state=42)
acc3_rf = evaluate(X, y_3_enc, groups, RandomForestClassifier,
                   "Random Forest", n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
acc3_gb = evaluate(X, y_3_enc, groups, GradientBoostingClassifier,
                   "Gradient Boost", n_estimators=200, max_depth=5, random_state=42)

print(f"\nChance level: {1/len(le_3.classes_):.3f} ({1/len(le_3.classes_):.1%})")
print(f"Best accuracy: {max(acc3_lr, acc3_svm, acc3_rf, acc3_gb):.3f}")

# ── Feature Importance ──
scaler_final = StandardScaler()
X_final = scaler_final.fit_transform(X)
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_final, y_encoded)

importances = rf_model.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]
print(f"\n{'='*60}")
print("TOP 10 FEATURES")
print("=" * 60)
for idx in top_idx:
    col_name = feature_cols[idx] if idx < len(feature_cols) else f"feature_{idx}"
    print(f"  {col_name}: {importances[idx]:.4f}")


Classes: ['angry', 'calm', 'happy', 'neutral', 'sad']
RESULTS — Keystroke Emotion Detection (5-class)
  LogReg (baseline): Acc=0.418±0.055  F1=0.438±0.060
  SVM (RBF): Acc=0.363±0.053  F1=0.379±0.061
  Random Forest: Acc=0.538±0.073  F1=0.468±0.071
  Gradient Boost: Acc=0.568±0.057  F1=0.536±0.052

Chance level: 0.200 (20.0%)
Best accuracy: 0.568

RESULTS — Keystroke Emotion Detection (3-class)
  LogReg (baseline): Acc=0.526±0.048  F1=0.536±0.054
  SVM (RBF): Acc=0.470±0.050  F1=0.478±0.057
  Random Forest: Acc=0.643±0.070  F1=0.628±0.068
  Gradient Boost: Acc=0.633±0.041  F1=0.621±0.040

Chance level: 0.333 (33.3%)
Best accuracy: 0.643

TOP 10 FEATURES
  n_keystrokes: 0.0701
  D1U2_median: 0.0289
  TotTime: 0.0287
  D1U2_q25: 0.0266
  D1U3_q25: 0.0257
  D1U1_q25: 0.0245
  D1U1_mean: 0.0239
  U1D2_q25: 0.0236
  D1U1_std: 0.0234
  D1U1_q75: 0.0227


In [3]:
# ═══════════════════════════════════════════════════════════
# CELL 3: DATASET LEAKAGE TEST
# ═══════════════════════════════════════════════════════════
# n_keystrokes (14%) and TotTime (6%) dominate feature importance.
# CONCERN: The model might be predicting session length/duration
# instead of actual emotion from typing patterns.
#
# If removing these features collapses accuracy, the model
# is "cheating" — using session metadata, not genuine emotion signals.
#
# We test 3 configurations:
#   A: All features (current)
#   B: Remove n_keystrokes + TotTime
#   C: Remove ALL session-level features (n_keystrokes, TotTime,
#      DelFreq, LeftFreq, error_ratio) — pure timing only

print("=" * 60)
print("LEAKAGE TEST — Is the model predicting emotion or session length?")
print("=" * 60)

# Identify which columns to remove
session_meta_cols = ['n_keystrokes', 'TotTime']
all_meta_cols = ['n_keystrokes', 'TotTime', 'DelFreq', 'LeftFreq', 'error_ratio']

session_meta_idx = [i for i, c in enumerate(feature_cols) if c in session_meta_cols]
all_meta_idx = [i for i, c in enumerate(feature_cols) if c in all_meta_cols]

print(f"\n  Model A: All {X.shape[1]} features (baseline)")
print(f"  Model B: Remove {session_meta_cols} → {X.shape[1] - len(session_meta_idx)} features")
print(f"  Model C: Remove {all_meta_cols} → {X.shape[1] - len(all_meta_idx)} features")

# Create feature subsets
keep_B = [i for i in range(X.shape[1]) if i not in session_meta_idx]
keep_C = [i for i in range(X.shape[1]) if i not in all_meta_idx]

X_B = X[:, keep_B]
X_C = X[:, keep_C]

# Test with best model (GB)
print(f"\n  Testing with Gradient Boosting (5-class):")
acc_A = evaluate(X, y_encoded, groups, GradientBoostingClassifier,
                 "Model A (all features)", n_estimators=200, max_depth=5, random_state=42)
acc_B = evaluate(X_B, y_encoded, groups, GradientBoostingClassifier,
                 "Model B (no n_keys/TotTime)", n_estimators=200, max_depth=5, random_state=42)
acc_C = evaluate(X_C, y_encoded, groups, GradientBoostingClassifier,
                 "Model C (pure timing)", n_estimators=200, max_depth=5, random_state=42)

print(f"\n  Testing with Gradient Boosting (3-class):")
acc3_A = evaluate(X, y_3_enc, groups, GradientBoostingClassifier,
                  "Model A (all features)", n_estimators=200, max_depth=5, random_state=42)
acc3_B = evaluate(X_B, y_3_enc, groups, GradientBoostingClassifier,
                  "Model B (no n_keys/TotTime)", n_estimators=200, max_depth=5, random_state=42)
acc3_C = evaluate(X_C, y_3_enc, groups, GradientBoostingClassifier,
                  "Model C (pure timing)", n_estimators=200, max_depth=5, random_state=42)

# Verdict
print(f"\n{'='*60}")
print("LEAKAGE VERDICT")
print("=" * 60)
print(f"  {'Config':<30} {'5-class':<10} {'3-class':<10}")
print(f"  {'-'*50}")
print(f"  {'A: All features':<30} {acc_A:<10.3f} {acc3_A:<10.3f}")
print(f"  {'B: No n_keys/TotTime':<30} {acc_B:<10.3f} {acc3_B:<10.3f}")
print(f"  {'C: Pure timing only':<30} {acc_C:<10.3f} {acc3_C:<10.3f}")
print(f"  {'Chance':<30} {'0.200':<10} {'0.333':<10}")

drop_5 = acc_A - acc_C
drop_3 = acc3_A - acc3_C
print(f"\n  Accuracy drop (A→C): 5-class={drop_5:+.3f}, 3-class={drop_3:+.3f}")

if drop_5 > 0.15:
    print("  ⚠️  SIGNIFICANT DROP — session metadata contributes heavily.")
    print("     Consider using Model B or C features for deployment.")
elif drop_5 > 0.05:
    print("  ⚡ MODERATE DROP — some signal from metadata, but timing patterns also contribute.")
    print("     Model B (remove n_keys/TotTime only) is a good compromise.")
else:
    print("  ✅ MINIMAL DROP — the model genuinely uses typing patterns, not session metadata.")
    print("     Safe to use all features.")


LEAKAGE TEST — Is the model predicting emotion or session length?

  Model A: All 50 features (baseline)
  Model B: Remove ['n_keystrokes', 'TotTime'] → 48 features
  Model C: Remove ['n_keystrokes', 'TotTime', 'DelFreq', 'LeftFreq', 'error_ratio'] → 45 features

  Testing with Gradient Boosting (5-class):
  Model A (all features): Acc=0.568±0.057  F1=0.536±0.052
  Model B (no n_keys/TotTime): Acc=0.495±0.042  F1=0.448±0.041
  Model C (pure timing): Acc=0.505±0.070  F1=0.467±0.066

  Testing with Gradient Boosting (3-class):
  Model A (all features): Acc=0.633±0.041  F1=0.621±0.040
  Model B (no n_keys/TotTime): Acc=0.543±0.067  F1=0.534±0.068
  Model C (pure timing): Acc=0.520±0.078  F1=0.512±0.074

LEAKAGE VERDICT
  Config                         5-class    3-class   
  --------------------------------------------------
  A: All features                0.568      0.633     
  B: No n_keys/TotTime           0.495      0.543     
  C: Pure timing only            0.505      0.520     
 

In [4]:
# ═══════════════════════════════════════════════════════════
# CELL 4: SAVE BEST MODEL WITH CALIBRATION
# ═══════════════════════════════════════════════════════════
# Based on leakage test results, choose which feature set to use.
# Then save calibrated model for trustworthy confidence scores.

# Decision logic: use Model B if n_keys/TotTime caused > 5% drop
# (meaning they were artificially inflating accuracy)
if acc_A - acc_B > 0.05:
    print("Using Model B features (removed n_keystrokes, TotTime)")
    X_save = X_B
    save_feature_cols = [c for c in feature_cols if c not in session_meta_cols]
    imputer_save = SimpleImputer(strategy='median')
    X_save = imputer_save.fit_transform(X_save)
else:
    print("Using all features (no significant leakage detected)")
    X_save = X
    save_feature_cols = feature_cols
    imputer_save = imputer  # use original imputer

print(f"Final feature count: {len(save_feature_cols)}")

# Train final GB model on all data
scaler_save = StandardScaler()
X_scaled = scaler_save.fit_transform(X_save)

base_gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, random_state=42
)
base_gb.fit(X_scaled, y_encoded)

# Calibrate probabilities
print("Calibrating probabilities...")
calibrated_gb = CalibratedClassifierCV(base_gb, method='isotonic', cv=5)
calibrated_gb.fit(X_scaled, y_encoded)

# Verify calibration
raw_proba = base_gb.predict_proba(X_scaled)
cal_proba = calibrated_gb.predict_proba(X_scaled)
print(f"\nProbability comparison (sample 0):")
print(f"  Raw GB:     {[f'{p:.3f}' for p in raw_proba[0]]}")
print(f"  Calibrated: {[f'{p:.3f}' for p in cal_proba[0]]}")

# Save
joblib.dump(calibrated_gb, "keystroke_emotion_model.pkl")
joblib.dump(scaler_save, "keystroke_scaler.pkl")
joblib.dump(le, "keystroke_label_encoder_5.pkl")
joblib.dump(le_3, "keystroke_label_encoder_3.pkl")
joblib.dump(imputer_save, "keystroke_imputer.pkl")
joblib.dump(save_feature_cols, "keystroke_feature_cols.pkl")

print(f"\n✅ Saved artifacts")


Using Model B features (removed n_keystrokes, TotTime)
Final feature count: 48
Calibrating probabilities...

Probability comparison (sample 0):
  Raw GB:     ['0.000', '0.000', '0.000', '1.000', '0.000']
  Calibrated: ['0.061', '0.050', '0.111', '0.687', '0.091']

✅ Saved artifacts
